# Évaluation d'un modèle

Une fois un modèle entraîné, une question fondamentale se pose : *« est-ce qu'il est bon ? »*. Mais **bon pour qui et pour quoi ?** Avant de dégainer une métrique, il faut toujours partir du **besoin métier**.

## Partir du besoin métier

- **Quel est l'objectif poursuivi ?**
  → Estimer les risques d'aggravation de la maladie à un an.
- **Comment l'organisation pourrait-elle bénéficier des prédictions de ce modèle ?**
  → En automatisant la prédiction (gain de temps, coût, disponibilité).
- **Quelle est la situation actuelle ?**
  → L'évaluation par des experts est coûteuse et aboutit à un pourcentage d'erreur de 40 % en moyenne.

> **Règle d'or :** une métrique ne vaut rien si elle n'est pas **comparée à une référence** (baseline, expert humain, modèle précédent...). Dire *« mon modèle fait 39% d'erreur »* ne veut rien dire en soi. Dire *« mon modèle fait 39% d'erreur alors que l'expert humain fait 40%, pour un coût 100× moindre »* est une phrase exploitable.

On dispose des données **Diabetes** de scikit-learn pour développer le modèle.

On identifie alors une métrique — par exemple le **pourcentage d'erreur moyen** de la prédiction, qu'il faut améliorer par rapport à la situation actuelle.

La variable à prédire est un **nombre** (un score quantitatif de progression de la maladie à un an), donc on a affaire à un problème de **régression**.

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

In [2]:
diabetes = load_diabetes()
print(diabetes.DESCR)

.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

  :Number of Instances: 442

  :Number of Attributes: First 10 columns are numeric predictive values

  :Target: Column 11 is a quantitative measure of disease progression one year after baseline

  :Attribute Information:
      - age     age in years
      - sex
      - bmi     body mass index
      - bp      average blood pressure
      - s1      tc, total serum cholesterol
      - s2      ldl, low-density lipoproteins
      - s3      hdl, high-density lipoproteins
      - s4      tch, total cholesterol / HDL
      - s5      ltg, possibly log of serum triglycerides level
      - s6      glu, blood sugar level

Note: Each of these 1

## Jeux de train et de test

**Pourquoi séparer ?** Si on évalue un modèle sur les mêmes données qu'il a vues pendant l'entraînement, on mesure seulement sa capacité à **mémoriser** — pas sa capacité à **généraliser** à de nouvelles données. C'est comme noter un élève en lui redonnant exactement les exercices qu'il a révisés : on apprend peu sur ce qu'il a réellement compris.

**La solution :** mettre de côté une partie des données *avant* l'entraînement (le jeu de **test**), et ne l'utiliser qu'**à la toute fin** pour mesurer les performances. Ici, on garde 33% des patients pour tester.

> L'évaluation sur le **test** est la seule qui donne une idée honnête de ce que fera le modèle en production, sur des patients qu'il n'a jamais vus.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.33, random_state=2
)

## Modèle

On entraîne un **arbre de décision de régression** (cf. notebook précédent) avec une profondeur limitée à 3 pour rester interprétable et éviter l'overfitting sur ce petit jeu (442 patients).

In [4]:
tree_reg = DecisionTreeRegressor(max_depth=3, random_state=3)
tree_reg.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=3, random_state=3)

## Évaluation du modèle

Pour évaluer la qualité d'un modèle, il nous faut une **métrique** qui mesure à quel point ses estimations sont justes — autrement dit, les **erreurs** commises par le modèle. Il en existe plusieurs ; on commence ici par la MAPE, qui est proche du langage métier (*« le modèle se trompe en moyenne de tant de pour cent »*).

### MAPE — Mean Absolute Percentage Error

$
\Large{
\text{MAPE}(y, \hat{y}) = \frac{1}{n_{\text{samples}}} \sum_{i=0}^{n_{\text{samples}}-1} \frac{\left| y_i - \hat{y}_i \right|}{\max(\epsilon, \left| y_i \right|)}
}
$

**Décodage, étape par étape :**

| Symbole | Ce que ça veut dire |
|---|---|
| $y_i$ | La vraie valeur pour le patient $i$ (ce que dit la réalité) |
| $\hat{y}_i$ | La valeur **prédite** par le modèle (le chapeau ^ sur $y$ se lit « y chapeau » ou « y estimé ») |
| $\|y_i - \hat{y}_i\|$ | L'**écart absolu** (toujours positif) entre réalité et prédiction |
| Division par $\|y_i\|$ | Transforme l'écart en **pourcentage** de la vraie valeur |
| $\max(\epsilon, \|y_i\|)$ | Astuce technique pour éviter une division par zéro si $y_i = 0$ ($\epsilon$ est un petit nombre > 0) |
| $\frac{1}{n} \sum$ | On fait la moyenne sur tous les patients |

**Interprétation :** la MAPE est une **moyenne de pourcentages d'erreur**. Une MAPE de 0,39 signifie *« en moyenne, le modèle se trompe de 39% sur la vraie valeur »*. Très parlant pour le métier.

**À retenir :**
- ✅ **Sans unité**, donc directement comparable entre problèmes (39% reste 39%, peu importe qu'on prédise des euros, des kg ou des scores).
- ⚠️ **À éviter si** $y$ peut valoir 0 ou être très petit : les erreurs relatives explosent.
- ⚠️ **Asymétrique** : une erreur de + et de - même grandeur ne comptent pas pareil en pourcentage (d'où l'existence de variantes comme sMAPE).

Voir __[Illustration: mape](http://digitalfirst.bfwpub.com/stats_applet/stats_applet_5_correg.html)__

In [5]:
from sklearn.metrics import mean_absolute_percentage_error

y_pred = tree_reg.predict(X_train)
mean_absolute_percentage_error(y_train, y_pred)

0.38649501083287485

In [6]:
y_pred = tree_reg.predict(X_test)
mean_absolute_percentage_error(y_test, y_pred)

0.3897923417337467

## Conclusion ?

Par rapport aux objectifs fixés :
- **Référence :** expert humain → 40% d'erreur.
- **Notre modèle :** MAPE train ≈ 39%, MAPE test ≈ 39%.
- **Écart train / test minime** → le modèle ne sur-apprend pas (pas d'overfitting visible à `max_depth=3`).

Le modèle se place **au niveau de l'expert humain** sur cette métrique, avec l'avantage d'être **automatisable et instantané**. C'est une première version qui peut justifier d'aller plus loin (essayer Random Forest, Gradient Boosting, feature engineering...). L'important est qu'on a désormais un **processus d'évaluation reproductible**, basé sur un jeu de test fixe et une métrique alignée sur le besoin métier.

## Autres métriques d'évaluation pour la régression

La MAPE n'est pas la seule manière de mesurer la qualité d'une régression. Chaque métrique a ses forces et ses faiblesses — on en regarde trois des plus classiques : **$R^2$**, **MAE** et **RMSE**. En pratique, on en calcule souvent plusieurs en parallèle pour se faire une idée complète.

Voir la liste complète : [Regression metrics (scikit-learn)](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).

### $R^2$ — Coefficient de détermination

$
\Large{
R^2(y, \hat{y}) = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2} = 1 - \frac{\text{Variance des erreurs}}{\text{Variance des données}}
}
$

**L'idée en une phrase :** le $R^2$ compare les erreurs de ton modèle à celles d'un **modèle bête qui prédirait toujours la moyenne** $\bar{y}$.

**Décodage :**
- Numérateur : $\sum (y_i - \hat{y}_i)^2$ → c'est la **somme des carrés des erreurs** de ton modèle.
- Dénominateur : $\sum (y_i - \bar{y})^2$ → c'est la **variance totale** des données (les erreurs qu'on ferait en prédisant toujours la moyenne).
- $1 - \frac{\text{erreurs modèle}}{\text{erreurs baseline}}$ → **quelle fraction de la variance des données ton modèle arrive-t-il à expliquer ?**

**Grille d'interprétation :**

| $R^2$ | Signification |
|---|---|
| $R^2 = 1$ | Modèle parfait, zéro erreur |
| $R^2 = 0$ | Modèle pas meilleur que « prédire la moyenne » — inutile |
| $R^2 < 0$ | Modèle **pire** que prédire la moyenne — clairement cassé |
| $R^2 \approx 0{,}8$ | Le modèle explique 80% de la variance — correct |

**⚠️ Piège classique :** le $R^2$ dépend de la **variance des données**. Sur un problème « facile » (données peu variables), un $R^2$ de 0,9 est banal ; sur un problème « difficile », un $R^2$ de 0,3 peut déjà être précieux. On ne compare donc pas des $R^2$ entre jeux de données différents.

Voir __[Illustration: $R^2$](http://digitalfirst.bfwpub.com/stats_applet/stats_applet_5_correg.html)__

In [7]:
from sklearn.metrics import r2_score

r2_score(y_test, y_pred)

0.33598713787710677

### MAE — Mean Absolute Error

$
\Large{
\text{MAE}(y, \hat{y}) = \frac{1}{n} \sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|
}
$

**L'idée en une phrase :** *« en moyenne, de combien mon modèle se trompe-t-il, dans les mêmes unités que la cible ? »*.

C'est sans doute la métrique la plus **intuitive** : on prend l'écart entre prédiction et réalité, on force à être positif avec la valeur absolue, et on fait la moyenne. Si la cible est en euros, la MAE est en euros. Si c'est en degrés, la MAE est en degrés.

**À retenir :**
- ✅ **Interprétation directe** dans les unités du problème → très bon pour discuter avec le métier.
- ✅ **Robuste aux outliers** : chaque erreur compte linéairement, peu importe sa taille (contrairement au RMSE).
- ⚠️ Pas dérivable en zéro (valeur absolue), ce qui complique l'optimisation directe — mais ce n'est un problème que pour certains algorithmes, pas pour l'évaluation.

Voir __[Illustration: mae](http://digitalfirst.bfwpub.com/stats_applet/stats_applet_5_correg.html)__

In [8]:
from sklearn.metrics import mean_absolute_error

mean_absolute_error(y_test, y_pred)

49.44855211803405

### RMSE — Root Mean Squared Error

$
\Large{
\text{RMSE}(y, \hat{y}) = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
}
$

**Décodage :**
1. On calcule l'écart de chaque prédiction : $(y_i - \hat{y}_i)$.
2. On l'**élève au carré** — ce qui a deux effets :
   - rendre l'écart positif (comme la valeur absolue) ;
   - **pénaliser les grosses erreurs beaucoup plus que les petites** (une erreur de 10 compte 100 fois plus qu'une erreur de 1).
3. On fait la **moyenne** de ces carrés.
4. On prend la **racine carrée** — pour retomber sur les unités d'origine de la cible (comme pour l'écart-type).

**Comparaison MAE vs RMSE :**
- Si **MAE ≈ RMSE**, les erreurs sont homogènes (peu de grosses erreurs isolées).
- Si **RMSE >> MAE**, le modèle fait quelques **grosses erreurs ponctuelles** qui font exploser le RMSE. C'est un signal que certains cas « craquent » — à investiguer (outliers ? classe minoritaire ? erreur de feature ?).

**À retenir :**
- ✅ **Même unité que la cible** (grâce à la racine), donc interprétable comme la MAE.
- ✅ **Dérivable partout**, ce qui en fait la fonction de perte favorite en optimisation (moindres carrés → régression linéaire).
- ⚠️ **Très sensible aux outliers** (à cause du carré). Si tes données ont des valeurs extrêmes, privilégier la MAE ou nettoyer avant.

Voir __[Illustration: rmse](http://digitalfirst.bfwpub.com/stats_applet/stats_applet_5_correg.html)__

In [9]:
from sklearn.metrics import mean_squared_error

rmse = mean_squared_error(y_test, y_pred, squared=False)
rmse

62.52977128443658

## 🎯 Quelle métrique choisir en pratique ?

Il n'y a **pas de métrique universellement meilleure** — chacune raconte une histoire différente sur les erreurs de ton modèle. Quelques repères :

### Tableau comparatif

| Métrique | Unité | Sensible aux outliers ? | Parle au métier ? | Quand l'utiliser ? |
|---|---|---|---|---|
| **MAE** | Celle de $y$ | Peu | ✅ directement | Référence fiable, erreurs homogènes |
| **RMSE** | Celle de $y$ | Beaucoup (carré) | ✅ | Quand les grosses erreurs sont graves à pénaliser |
| **MAPE** | % | Moyennement | ✅✅ (en %) | Quand le métier raisonne en pourcentages, si $y \neq 0$ |
| **$R^2$** | Sans unité | Moyennement | ⚠️ plus abstrait | Pour comparer des modèles sur le **même** jeu de données |

### Réflexes en pratique

- **Toujours** avoir une **baseline** : « prédire la moyenne », « prédire la valeur d'hier », ou un modèle trivial. Une métrique absolue ne dit rien ; la comparaison à une baseline, si.
- **Calculer plusieurs métriques** simultanément — elles se complètent.
- Si **RMSE >> MAE** → aller voir les prédictions où le modèle craque ; il y a probablement des outliers ou une classe mal apprise.
- **Choisir la métrique AVANT l'entraînement**, et s'y tenir. Changer de métrique en cours de route pour rendre les chiffres plus flatteurs est un classique... à éviter absolument.
- **Aligner la métrique sur le coût métier d'une erreur.** Prédire le prix d'un billet d'avion avec +50€ vs -50€ d'erreur n'a pas forcément le même coût côté métier → il existe aussi des métriques asymétriques (pinball loss, quantile loss...).

### À retenir

> Une métrique n'est pas une vérité absolue, c'est un **résumé** — et comme tout résumé, elle cache des choses. Regarde toujours **plusieurs métriques + la distribution des erreurs** (histogramme, résidus, cas les plus pénalisés) avant de déclarer qu'un modèle est bon.